In [36]:
import task1
import numpy as np

### Main Task 1: Signal Acquisition

In [37]:
freq = 20

sigA, sigB = task1.generate_signals(freq)

Shift Samples: -11


### Main task 2: DFT Compute

In [38]:
def compute_dft(sig: np.ndarray) -> np.ndarray:
    
    N = len(sig)
    
    dft = np.zeros_like(sig, dtype=np.complex128)
    
    for k in range(N):
        ex = np.zeros_like(dft)
        
        for n in range(N):
            ex[n] = sig[n] * np.exp(-1j * 2 * np.pi * (1/N) * k * n)
            
        dft[k] = np.sum(ex)
    
    
      
    return dft
    
    
# Alt:

# def compute_dft(sig: np.ndarray) -> np.ndarray:
#     N = len(sig)
#     dft = np.zeros(N, dtype=np.complex128)
    
#     for k in range(N):
#         total = 0.0 + 0.0j
#         for n in range(N):
#             total += sig[n] * np.exp(-1j * 2 * np.pi * k * n / N)
#         dft[k] = total
        
#     return dft

# def compute_dft(sig: np.ndarray) -> np.ndarray:
#     N = len(sig)
#     n = np.arange(N)
#     dft = np.zeros(N, dtype=np.complex128)
    
#     for k in range(N):
#         dft[k] = np.sum(sig * np.exp(-1j * 2 * np.pi * k * n / N))
        
#     return dft


In [39]:
def compute_idft(dft: np.ndarray) -> np.ndarray:
    N = len(dft)
    k = np.arange(N)
    
    sig = np.zeros_like(dft, dtype=np.complex128)
    
    for n in range(N):
        sig[n] = np.sum(dft * np.exp(1j * 2 * np.pi * n * (1/N) * k))
    
    return sig

In [40]:
dftA = np.real(compute_dft(sigA))
dftB = np.real(compute_dft(sigB))

## Cross-Correlation: The Concept, Theory, and Implementation

### 1. What is Cross-Correlation? (The Concept)

At its core, cross-correlation is a measure of similarity between two signals as one is shifted (delayed or advanced) relative to the other.

Imagine tracing Signal A onto a piece of transparent paper and sliding it horizontally over Signal B.

* At each shift (or "lag"), you multiply the overlapping values and sum them up.
* If the peaks and valleys align perfectly, the resulting sum is a large positive number.
* If a peak aligns with a valley, the sum is a large negative number.
* If they are completely out of sync (or just random noise), the sum is close to zero.

In the context of your assignment's scenario, Station A and Station B record the same seismic event, but Station B's recording is delayed and noisy. Cross-correlation acts as a pattern-matching tool to find the exact time lag where the two seismic waves best align, which reveals the physical delay between the stations.

### 2. How to Calculate It Theoretically

There are two distinct ways to compute this, which highlights exactly why the Fast Fourier Transform (FFT) is such a powerful algorithm.

**Method A: The Time-Domain Approach (Direct Calculation)**
Mathematically, the discrete cross-correlation involves shifting one signal over another and calculating the sum of their point-wise products at each shift. For signals $x$ and $y$, the cross-correlation at lag $m$ is:


$$R_{xy}(m) = \sum_{n=0}^{N-1} x(n) \cdot y(n-m)$$


*The Problem:* For a signal of length $N$, you have to perform $N$ multiplications and additions for $N$ different shifts. This results in a computational complexity of $O(N^2)$. For massive audio or seismic files, this is far too slow.

**Method B: The Frequency-Domain Approach (The DFT Way)**
This is the approach required for your assignment. The Cross-Correlation Theorem states that time-domain cross-correlation is equivalent to complex multiplication in the frequency domain.
Instead of sliding the signals in the time domain, you:

1. Transform both signals into the frequency domain using DFT.


2. Take the complex conjugate of the second signal's DFT.
3. Multiply them point-by-point.
4. Transform the result back to the time domain using IDFT.



### 3. How to Implement It in Your Project

Based on the problem description, you are required to use your custom DFT and IDFT functions without relying on built-in packages.

Here is the step-by-step logic you need to write in Python:

* **Step 1: Compute DFTs**
Pass both arrays through the `compute_dft` function you just built.
```python
dft_A = compute_dft(sigA)
dft_B = compute_dft(sigB)

```


* **Step 2: Conjugate and Multiply**
You need to multiply `dft_A` by the complex conjugate of `dft_B`. You can use NumPy's `np.conjugate()` for this step.
```python
cross_spectrum = dft_A * np.conjugate(dft_B)

```


* **Step 3: Compute IDFT and Extract Real Part**
Pass the resulting `cross_spectrum` through your `compute_idft` function. Because of minor floating-point inaccuracies during the complex math, the result will have a tiny imaginary component. The assignment specifically dictates that you must consider only the real part of the IDFT result.


```python
cross_corr = compute_idft(cross_spectrum).real

```


* **Step 4: Find the Lag (The Tricky Part)**
The DFT assumes signals are periodic (circular). Therefore, the raw `cross_corr` array doesn't go from $-N/2$ to $+N/2$ directly. Instead:
* Index `0` represents a lag of 0.
* Indices `1` to `N//2` represent positive lags.
* Indices `N//2 + 1` to `N-1` represent negative lags (wrapped around backwards).


To find the actual shift:
1. Find the index of the maximum positive value in the `cross_corr` array using `np.argmax()`.


2. If that index is greater than $N/2$, you must subtract $N$ to get the true negative lag value.


* **Step 5: Estimate the Distance**
Once you have your true integer lag, plug it into the assignment's formula:



$$\text{Distance} = \vert{}\text{Sample lag}\vert{} \times \left(\frac{1}{\text{Sampling rate}}\right) \times \text{Wave velocity}$$



*(Note: use the `sampling_rate` of 100 and `wave_velocity` of 8000 provided in the boilerplate code)*.



Does the logic for extracting the negative lags from the wrapped array make sense, or would you like to see a concrete example of how the array indices map to the lag values?